In [ ]:
import torch
from linops import LinearOperator
from torch.func import jvp

from rlaopt.expression import Expression, Variable
from rlaopt.ext_tensordict import TensorDict

In [ ]:
x = Variable(torch.ones(3))
y = Variable(torch.zeros(2))

A = torch.randn(4, 3)
B = torch.randn(4, 2)
c = torch.randn(4)

In [ ]:
expr = A @ x + B @ y + A @ x + c

In [ ]:
expr.is_affine()

### Extract bias by evaluating at zero

In [ ]:
bias = expr.evaluate(torch.zeros_like(expr.variable_values))

In [ ]:
bias

In [ ]:
c

Try to get LinearOperator working

In [ ]:
class AffineExprLinOp(LinearOperator):
    def __init__(
        self, affine_expr: Expression, bias: torch.Tensor, smooth_expr_vars: TensorDict
    ):
        super().__init__()
        self._affine_expr = affine_expr
        self._bias = bias
        self._unflatten = lambda v: smooth_expr_vars.from_flat_tensor(v)

        input_dim = smooth_expr_vars.flat_dim()
        self._shape = (self._bias.shape[0], input_dim)
        self._device = self._bias.device

    def _matmul_impl(self, v: torch.Tensor):
        v_td = self._unflatten(v)
        return self._affine_expr.evaluate(v_td) - self._bias

In [ ]:
class AffineExprLinOp(LinearOperator):
    def __init__(self, affine_expr: Expression, smooth_expr_vars: TensorDict):
        super().__init__()
        self._affine_expr = affine_expr
        self._zero_vars = smooth_expr_vars.apply(torch.zeros_like)  # Cache zero point

        input_dim = smooth_expr_vars.flat_dim()
        output = affine_expr.evaluate(self._zero_vars)
        self._shape = (output.shape[0], input_dim)
        self._device = output.device

    def _matmul_impl(self, v: torch.Tensor):
        """Compute A @ v using JVP."""
        v_td = self._zero_vars.from_flat_tensor(v)

        # JVP computes d/dt f(x + t*v) at t=0, which gives Jacobian @ v = A @ v
        _, jvp_result = jvp(
            lambda vars: self._affine_expr.evaluate(vars), (self._zero_vars,), (v_td,)
        )
        return jvp_result

In [ ]:
# lin_op = AffineExprLinOp(expr, bias, expr.variable_values)
lin_op = AffineExprLinOp(expr, expr.variable_values)

In [ ]:
lin_op.shape

In [ ]:
lin_op.T.shape

In [ ]:
lin_op @ torch.eye(lin_op.shape[1])

In [ ]:
torch.hstack([2 * A, B])

In [ ]:
lin_op.T @ torch.eye(lin_op.shape[0])